Objective: Create meaningful new features based on the available data.

In [1]:
import pandas as pd

# Load the cleaned data, ensuring the date column is parsed correctly
df = pd.read_csv('../data/processed/cleaned_orders.csv', parse_dates=['order_date_dateorders'])

In [2]:
# 1. Shipping Delay (instead of delivery delay)
# This calculates if the shipment was early, on time, or late.
df['shipping_delay'] = df['days_for_shipping_real'] - df['days_for_shipment_scheduled']

# 2. Profit Margin Ratio
# We can calculate this from 'benefit_per_order' and 'sales_per_customer'
# To avoid division by zero, we replace 0 sales with a small number (e.g., 1)
df['profit_margin_ratio'] = df['benefit_per_order'] / df['sales_per_customer'].replace(0, 1)

# 3. Extract Time-Based Features from the order date
df['order_year'] = df['order_date_dateorders'].dt.year
df['order_month'] = df['order_date_dateorders'].dt.month
df['order_weekday'] = df['order_date_dateorders'].dt.dayofweek

# 4. A perfect order is on time (late_delivery_risk == 0) and profitable (benefit_per_order > 0)
df['is_perfect_order'] = ((df['late_delivery_risk'] == 0) & (df['benefit_per_order'] > 0)).astype(int)

# 4. Is the order on a weekend?
df['is_weekend'] = df['order_weekday'].apply(lambda x: 1 if x >= 5 else 0)

# 5. Profit per day scheduled (is this a high-value, urgent order?)
# Use 1 to avoid division by zero
df['profit_per_day_scheduled'] = df['benefit_per_order'] / (df['days_for_shipment_scheduled'] + 1)


# --- Historical/Group Features ---

# Calculate the average late rate for each category
category_late_rate = df.groupby('category_name')['late_delivery_risk'].mean().to_dict()
df['category_late_rate'] = df['category_name'].map(category_late_rate)

# Calculate the average late rate for each customer
customer_late_rate = df.groupby('customer_id')['late_delivery_risk'].mean().to_dict()
df['customer_late_rate'] = df['customer_id'].map(customer_late_rate)

# Fill any new customers/categories with the overall average
overall_late_rate = df['late_delivery_risk'].mean()
df['category_late_rate'].fillna(overall_late_rate, inplace=True)
df['customer_late_rate'].fillna(overall_late_rate, inplace=True)


C:\Users\dombl\AppData\Local\Temp\ipykernel_6376\4050310333.py:38: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['category_late_rate'].fillna(overall_late_rate, inplace=True)
C:\Users\dombl\AppData\Local\Temp\ipykernel_6376\4050310333.py:39: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as 

In [3]:
df.to_csv('../data/processed/final_features.csv', index=False)
print("Feature-engineered data saved successfully to 'data/processed/final_features.csv'")

Feature-engineered data saved successfully to 'data/processed/final_features.csv'
